In [2]:
import os, copy, argparse, numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score

In [11]:
from main import GraphConvolution, GCN_Encoder, Classifier, VCDN, prepare_data, generate_adj_mat, load_model_dict, calculate_sample_weight

In [4]:
data_dir = './data/BRCA'
output_dir = './result/BRCA'
if 'ROSMAP' in data_dir:
    num_class = 2; adj_avg_degree = 2; hidden_dim_list = [200, 200, 100]; num_epoch_pretrain = 500; num_epoch = 2500; lr_e_pretrain = 1e-3; lr_e = 5e-4; lr_c = 1e-3
if 'BRCA' in data_dir:
    num_class = 5; adj_avg_degree = 10; hidden_dim_list = [400, 400, 200]; num_epoch_pretrain = 500; num_epoch = 2500; lr_e_pretrain = 1e-3; lr_e = 5e-4; lr_c = 1e-3

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [18]:
# Load Data.
data_train_list, data_test_list, data_all_list, label_train, label_test, label_all = prepare_data(data_dir=data_dir)
adj_train_list, adj_test_list = generate_adj_mat(data_train_list, data_test_list, adj_avg_degree)
label_train = torch.tensor(label_train, dtype=torch.long).to(device)
onehot_label_train = F.one_hot(label_train, num_class).to(device)
label_test = torch.tensor(label_test, dtype=torch.long).to(device)
label_test = label_test.cpu().numpy()
label_all = torch.tensor(label_all, dtype=torch.long).to(device)
for i in range(len(data_train_list)):
    data_train_list[i] = torch.tensor(data_train_list[i], dtype=torch.float).to(device)
    data_test_list[i] = torch.tensor(data_test_list[i], dtype=torch.float).to(device)
    data_all_list[i] = torch.tensor(data_all_list[i], dtype=torch.float).to(device)
    adj_train_list[i] = torch.tensor(adj_train_list[i], dtype=torch.float).to_sparse().to(device)
    adj_test_list[i] = torch.tensor(adj_test_list[i], dtype=torch.float).to_sparse().to(device)

In [7]:
num_view = len(data_train_list)
data_train_dim_list = [x.shape[1] for x in data_train_list]
featname_list = []
for i in range(num_view):
    df = pd.read_csv(os.path.join(data_dir, str(i+1) + '_featname.csv'), header=None)
    featname_list.append(df.values.flatten())
# featname_list

In [20]:
data_train_dim_list

[1000, 1000, 503]

In [ ]:
featimp_list_list = []
for rep in range(5):
    model_dict = {}
    for i in range(num_view):
        model_dict["E{:}".format(i+1)] = GCN_Encoder(data_train_dim_list[i], hidden_dim_list, 0.5)
        model_dict["C{:}".format(i+1)] = Classifier(hidden_dim_list[-1], num_class)
    if num_view >= 2:
        model_dict["C"] = VCDN(num_view, num_class, pow(num_class, num_view))
    model_dict = load_model_dict(os.path.join(output_dir, '{:d}'.format(rep+1)), model_dict)
    
    for m in model_dict:
        model_dict[m].to(device)
    
    for m in model_dict:
        model_dict[m].eval()
        
    output_list = []
    for i in range(num_view):
        output_list.append(model_dict["C{:}".format(i+1)](model_dict["E{:}".format(i+1)](data_all_list[i],adj_test_list[i])))
    output = model_dict["C"](output_list) if num_view >= 2 else output_list[0]   
    num_train = len(data_train_list[0]) 
    output = output[num_train:,:]
    prob = F.softmax(output, dim=1).data.cpu().numpy()
    f1 = f1_score(label_test, prob.argmax(1)) if num_class == 2 else f1_score(label_test, prob.argmax(1), average='macro')
    
    feat_imp_list = [] # shape: (num_view)
    for i in range(num_view):
        feat_imp = {"feat_name": featname_list[i], "feat_imp": np.zeros(len(featname_list[i]))}
        for j in range(len(featname_list[i])):
            feat_train = data_train_list[i][:,j].clone()
            feat_all = data_all_list[i][:,j].clone()
            data_train_list[i][:,j] = 0
            data_all_list[i][:,j] = 0
            adj_train_list, adj_test_list = generate_adj_mat(data_train_list, data_test_list, adj_avg_degree)
            adj_train_list[i] = torch.tensor(adj_train_list[i], dtype=torch.float).to_sparse().to(device)
            adj_test_list[i] = torch.tensor(adj_test_list[i], dtype=torch.float).to_sparse().to(device)
            
            output_list = []
            for k in range(num_view):
                output_list.append(model_dict["C{:}".format(k+1)](model_dict["E{:}".format(k+1)](data_all_list[k], adj_test_list[k])))
            output = model_dict["C"](output_list) if num_view >= 2 else output_list[0]  
            num_train = len(data_train_list[0])
            output = output[num_train:,:]
            prob = F.softmax(output, dim=1).data.cpu().numpy()
            f1_tmp = f1_score(label_test, prob.argmax(1)) if num_class == 2 else f1_score(label_test, prob.argmax(1), average='macro')
            
            feat_imp['feat_imp'][j] = (f1-f1_tmp) * data_train_dim_list[i]
            data_train_list[i][:,j] = feat_train.clone()
            data_all_list[i][:,j] = feat_all.clone()
            
        feat_imp_list.append(pd.DataFrame(data=feat_imp))
    
    featimp_list_list.append(copy.deepcopy(feat_imp_list)) # shape: (num_rep, num_view)

In [46]:
# featimp_list_list shape: (num_rep, num_view)
topn=30
num_rep = len(featimp_list_list)
num_view = len(featimp_list_list[0])

df_tmp_list = []
for r in range(0, num_rep):
    for v in range(num_view):
        df_tmp = copy.deepcopy(featimp_list_list[r][v])
        df_tmp['omics'] = np.ones(df_tmp.shape[0], dtype=int) * v
        df_tmp_list.append(df_tmp.copy(deep=True))
df_featimp = pd.concat(df_tmp_list).copy(deep=True)
df_featimp = df_featimp.groupby(['feat_name', 'omics'])['feat_imp'].sum()
df_featimp = df_featimp.reset_index()
df_featimp = df_featimp.sort_values(by='feat_imp', ascending=False)
df_featimp_top = df_featimp.iloc[:topn]

print('Rank \t Feature name'.format('Rank','Feature name'))
for i in range(len(df_featimp_top)):
    print('{:} \t {:}'.format(i+1, df_featimp_top.iloc[i]['feat_name']))

Rank 	 Feature name
1 	 MORN3
2 	 OR1J4
3 	 AGR3
4 	 OR8A1
5 	 CT62
6 	 GPR37L1
7 	 RUSC2
8 	 DOK5
9 	 TACC1
10 	 COLQ
11 	 POU3F3
12 	 LOC645638
13 	 HOXD11
14 	 GAS1
15 	 SOX21
16 	 LOC100130264
17 	 SNORD93
18 	 CAPN13|92291
19 	 SIDT1|54847
20 	 GJB3|2707
21 	 RIMBP3B
22 	 TM4SF18
23 	 TMEM207
24 	 LMOD3
25 	 LOC84740|84740
26 	 LHX1
27 	 REEP6|92840
28 	 LEMD1|93273
29 	 LMX1B|4010
30 	 ZER1|10444
